In [1]:
import random
import json

#### JSON from the editor
```json
{
  "nodes": [
    {
      "width": 240,
      "height": 276,
      "id": "0.05272377425322505-1756500835216",
      "type": "eventType",
      "position": {
        "x": 145.53354881065553,
        "y": 1196.7836919286149
      },
      "data": {
        "box": {
          "newEventVars": {
            "0": [
              "payment reminder"
            ],
            "1": [
              "pay order"
            ]
          },
          "newObjectVars": {
            "0": [
              "orders"
            ]
          },
          "filters": [
            {
              "type": "O2E",
              "object": 0,
              "event": 0,
              "qualifier": null
            },
            {
              "type": "O2E",
              "object": 0,
              "event": 1,
              "qualifier": null
            }
          ],
          "sizeFilters": [],
          "constraints": [
            {
              "type": "Filter",
              "filter": {
                "type": "TimeBetweenEvents",
                "from_event": 0,
                "to_event": 1,
                "min_seconds": 0,
                "max_seconds": null
              }
            }
          ],
          "evVarLabels": {},
          "obVarLabels": {}
        }
      },
      "positionAbsolute": {
        "x": 145.53354881065553,
        "y": 1196.7836919286149
      },
      "selected": true,
      "dragging": false
    }
  ],
  "edges": []
}
```

#### JSON given to the DB
```json
{
  "tree": {
    "nodes": [
      {
         "Box": [
           {
             "newEventVars": { "0": ["payment reminder"], "1": ["pay order"] },
             "newObjectVars": { "0": ["orders"] },
             "filters": [
               { "type": "O2E", "object": 0, "event": 0, "qualifier": null },
               { "type": "O2E", "object": 0, "event": 1, "qualifier": null }
             ],
             "sizeFilters": [],
             "constraints": [
               {
                 "type": "Filter",
                 "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": null }
               }
             ],
             "evVarLabels": {},
             "obVarLabels": {}
           },
           [] // IDs of any connected child nodes in the tree
         ]
      }
    ],
    "edgeNames": []
  },
  "database": "DuckDB", // Target Database Engine (if doing DB translation)
  "table_mappings": { ... }
}
```

In [ ]:
patterns = [
    "EVENT_A happens before EVENT_B",
    "Whenever EVENT_A occurs, EVENT_B eventually occurs",
    "After EVENT_A, EVENT_B must not occur",
    "EVENT_A and EVENT_B refer to the same OBJECT",
    "EVENT_B happens within TIME hours after EVENT_A"
]

domains = {
    "e-commerce": {
        "events": ["create order", "add item", "pay order", "ship package", "cancel order"],
        "objects": ["customers", "employees", "items", "orders", "packages", "products"]
    },
    "recruitment": {
        "events": ["submit application", "review application", "approve application", "reject application"],
        "objects": ["applications", "candidates"]
    },
    "software_dev": {
        "events": ["open issue", "submit pull request", "review pull request", "merge pull request", "close issue"],
        "objects": ["issues", "pull requests", "commits"]
    },
    "distribution": {
        "events": ["Apply Discount", "Apply Payment", "Approve Order", "Approve Return", "Generate Invoice", 
                   "Generate Pick List", "Generate Shipping Label", "Handle Payment Discrepancy", 
                   "Issue Credit Note", "Issue Payment Reminder", "Pack Items", "Perform Credit Check", 
                   "Pick Items", "Process Return Request", "Receive Customer Order", "Receive Payment", 
                   "Reject Order", "Schedule Order Fulfillment", "Send Invoice to Customer", "Ship Order", 
                   "Update Inventory", "Validate Customer Order"],
        "objects": ["CreditNote", "Customer", "Dispute", "Invoice", "Order", "Payment", "Product", "Return", "Shipment"]
    },
    "procurement": {
        "events": ["Approve PO", "Approve PR", "Create PO", "Create PO (Batch)", "Create PR", "Make Payment", 
                   "Receive Goods", "Receive Invoice", "Reject PR", "Send PO to Supplier", 
                   "Three-way Match - Mismatch Found", "Three-way Match - Successful"],
        "objects": ["Goods Receipt", "Purchase Requisition", "Payment", "Invoice", "Purchase Order", "Supplier"]
    },
    "02_p2p": {
        "events": ["Approve PO", "Approve PR", "Create PO", "Create PO (Batch)", "Create PR", "Make Payment", 
                   "Receive Goods", "Receive Invoice", "Reject PR", "Send PO to Supplier", "Three-way Match - Mismatch Found", 
                   "Three-way Match - Successful"],
        "objects": ["Goods Receipt", "Invoice", "Payment", "Purchase Order", "Purchase Requisition", "Supplier"]
    },
    "hiring": {
        "events": ["Application Rejected", "Backend Interview", "Background Check", "Behavioral Interview", 
                   "Coding Interview", "Extensive Background Check", "Frontend Interview", 
                   "Hand In Job Application", "ML Interview", "Make Job Offer", "Screening Interview", "Telephonic Screening"],
        "objects": ["Age", "Case", "Person", "Yearsofeducation", "isCitizen", "isGender", "isGermanspeaking", 
                    "isReligious", "numOcc_ApplicationRejected", "numOcc_BackendInterview", "numOcc_BackgroundCheck", 
                    "numOcc_BehavioralInterview", "numOcc_CodingInterview", "numOcc_ExtensiveBackgroundCheck", 
                    "numOcc_FrontendInterview", "numOcc_HandInJobApplication", "numOcc_MLInterview", 
                    "numOcc_MakeJobOffer", "numOcc_ScreeningInterview", "numOcc_TelephonicScreening", 
                    "overallProtected", "reworkedActivities", "sumBoolDiscrFactors", "throughputTime"]
    },
    # not sure about the events and objects here, but can be added later
    # "angular_github_commits": {
    #     "events": ["add", "address", "aio", "benchmark", "bug", "build", "bump", "check", "chore", "ci", "cleanup", "code", "condition", "configure", "consolidated", "copy", "design", "developer.md", "docs", "don't", "edit", "example", "examples", "feat", "filetree", "finished", "first", "fix", "form", "format", "include", "initial", "keep", "make", "merge", "misc", "more", "moved", "patch", "perf", "produce", "read", "readme", "refactor", "release", "remove", "rendererv2", "repackaging", "replaced", "resolve", "revert", "roll", "router", "squash", "squash-me", "style", "table", "temporary", "test", "text", "tools", "transpiler", "tweak", "typo", "update", "use", "wip"],
    #     "objects": ["branches", "files"]
    # },
    "hospital": {
        "events": ["Diagnosis", "Discharge", "Examination", "Expert Examination", "Register at ER", "Register at FD", 
                   "Thorough Examination", "Treatment", "Treatment successful", "Treatment unsuccessful"],
        "objects": ["Age", "Case", "Person", "isCitizen", "isGender", "isGermanspeaking", "isPrivate_insurance", 
                    "isUnderlying_condition", "numOcc_Diagnosis", "numOcc_Discharge", "numOcc_Examination", 
                    "numOcc_ExpertExamination", "numOcc_RegisteratER", "numOcc_RegisteratFD", "numOcc_ThoroughExamination", 
                    "numOcc_Treatment", "numOcc_Treatmentsuccessful", "numOcc_Treatmentunsuccessful", "overallProtected", 
                    "reworkedActivities", "sumBoolDiscrFactors", "throughputTime"]
    },
    "ocel2-p2p": {
        "events": ["Approve Purchase Order", "Approve Purchase Requisition", "Create Goods Receipt", "Create Invoice Receipt", 
                   "Create Purchase Order", "Create Purchase Requisition", "Create Request for Quotation", 
                   "Delegate Purchase Requisition Approval", "Execute Payment", "Perform Two-Way Match", "lifecycle", "resource"],
        "objects": ["Amount (DMBTR)", "Credit Amount (BSEG-WRBTR)", "Debit Amount (BSEG-DMBTR)", 
                    "Delivery Date (EKPO-BEDAT)", "Document Type (EKKO-BSART)", "Invoice Receipt (MSEG-WEAHR)", 
                    "Material (EKPO-MATNR)", "Movement Type (MSEG-BWART)", "Net Price (EKPO-NETPR)", 
                    "Payment Block (RSEG-ZLSPR)", "Payment Method (ZLSCH)", "Plant (EKPO-WERKS)", 
                    "Posting Date (MKPF-BLDAT)", "Purchasing Group (EBAN-EKGRP)", "Purchasing Group (EKKO-EKGRP)", 
                    "Purchasing Organization (EBAN-EKORG)", "Purchasing Organization (EKKO-EKORG)", 
                    "Quantity (EKPO-MENGE)", "RFQ Type (EBAN-BSART)", "Release Indicator (EBAN-FRGZU)", 
                    "Release Status (EKKO-FRGZU)", "Storage Location (EKPO-LGORT)", "Vendor (EBAN-LIFNR)", 
                    "Vendor (EKKO-LIFNR)", "goods receipt", "invoice receipt", "material", "new_value", 
                    "payment", "purchase_order", "purchase_requisition", "quotation"]
    },
    # no. just no...
    # "Period20_procedure_steps_all_additional_event_types": {
    #     "events": ["1. Beratung", "1. Beratung (Einzelpl\u00e4ne)", "1. Beratung (Gesetzentwurf)", "1. Durchgang", 
    #                "2. Beratung", "2. Beratung (Einzelpl\u00e4ne)", "2. Beratung (Gesetzentwurf)", 
    #                "2. Beratung und Schlussabstimmung", "2. Durchgang", "3. Beratung", "Absetzung von der TO", 
    #                "Abstimmung", "Abstimmung \u00fcber Vermittlungsvorschlag", "Aktuelle Stunde", 
    #                "Allgemeine Verwaltungsvorschrift", "Amtliche Mitteilung: R\u00fccknahme", "Amts\u00fcbernahme", 
    #                "Ansprache", "Antrag", "Antrag auf Anrufung des Vermittlungsausschusses", 
    #                "Antrag auf Einsetzung einer Enquete-Kommission", "Antrag auf Einsetzung eines Untersuchungsausschusses", 
    #                "Antrag auf Entlastung der BRg", "Antrag auf Entschlie\u00dfung", "Antrag auf erneute Behandlung", 
    #                "Antrag auf erneute \u00dcberweisung", "Antrag des Pr\u00e4sidenten des BRH", 
    #                "Antrag zur Arbeit der Enquete-Kommission", "Antrag zur Gesch\u00e4ftsordnung", 
    #                "Antrag zur Weitergeltung von Gesch\u00e4ftsordnungsrecht", "Antwort", "Ausfertigung", 
    #                "Ausschusszuweisung", "Ausschuss\u00fcberweisung", "Aussprache", "BR-Sitzung", 
    #                "Bekanntgabe der Bildung der Bundesregierung und Eidesleistung der Bundesminister", 
    #                "Bekanntgabe gem\u00e4\u00df \u00a7 23 Abs. 1 Gesch\u00e4ftsordnung BR", 
    #                "Bemerkungen des Bundesrechnungshofes", "Beratung", "Beratung des Antrags auf erneute Behandlung", 
    #                "Beratung des Berichts \u00fcber den Stand der Ausschussberatung", 
    #                "Beratung \u00fcber Antrag auf erneute Behandlung", 
    #                "Beratung \u00fcber Bericht gem\u00e4\u00df \u00a7 62 Abs. 2 Gesch\u00e4ftsordnung BT", "Bericht", 
    #                "Bericht der Kommission", "Bericht gem\u00e4\u00df \u00a7 96 Gesch\u00e4ftsordnung BT", 
    #                "Bericht \u00fcber den Stand der Ausschussberatung gem\u00e4\u00df \u00a7 62 Abs. 2 Gesch\u00e4ftsordnung BT", 
    #                "Berichtigung", "Berichtigung zum Beschluss", "Berichtigung zum Gesetzesbeschluss", "Beschlussdrucksache", 
    #                "Beschlussdrucksache / Verordnungsentwurf", "Beschlussempfehlung", 
    #                "Beschlussempfehlung des Wahlpr\u00fcfungsausschusses", "Beschlussempfehlung und Bericht", 
    #                "Beschlussempfehlung und Bericht betreffend Wahleinspr\u00fcche", 
    #                "Beschlussempfehlung und Bericht des Untersuchungsausschusses", "Durchgang", "Einigungsvorschlag", 
    #                "Einleitende Ausf\u00fchrungen und Befragung der Bundesregierung", "Empfehlungen der Aussch\u00fcsse", 
    #                "Erg\u00e4nzende Antwort", "Erg\u00e4nzende schriftliche Antwort", "Erg\u00e4nzung zum Gesetzentwurf", 
    #                "Erkl\u00e4rung", "Erkl\u00e4rung der BRg", 
    #                "Erneute \u00dcberweisung gem\u00e4\u00df \u00a7 93 Gesch\u00e4ftsordnung BT", "Feststellung der Tagesordnung", 
    #                "Fortsetzung: 1. Beratung", "Fortsetzung: 2. Beratung", "Fortsetzung: Abstimmung", "Fortsetzung: Aussprache", 
    #                "Fortsetzung: Beratung", "Frage zum Grund der Frist\u00fcberschreitung", 
    #                "Gegen\u00e4u\u00dferung der BRg zur Stellungnahme des BR", 
    #                "Gesch\u00e4ftsordnungsantrag auf Absetzung der 1. Beratung von der TO", 
    #                "Gesch\u00e4ftsordnungsantrag auf Absetzung von der TO", "Gesch\u00e4ftsordnungsantrag auf Aufsetzung auf die TO", 
    #                "Gesch\u00e4ftsordnungsantrag auf Aufsetzung der 1. Beratung auf die TO", "Gesch\u00e4ftsordnungsantrag zur Beratung", 
    #                "Gesetzentwurf", "Gesetzesantrag", "Gro\u00dfe Anfrage", "Inkrafttreten", "Kleine Anfrage", "L\u00e4nderantrag", 
    #                "Mitteilung", "Mitteilung des federf\u00fchrenden Ausschusses", "Mitteilung zur Subsidiarit\u00e4tsfrist", 
    #                "M\u00fcndliche Frage", "M\u00fcndliche Frage/M\u00fcndliche Antwort", "M\u00fcndliche Frage/Schriftliche Antwort", 
    #                "Nachtrag: 2. Beratung", "Nachtrag: 2. Beratung und Schlussabstimmung", "Nachtrag: 3. Beratung", 
    #                "Nachtrag: Abstimmung \u00fcber Vermittlungsvorschlag", "Nachtrag: Antrag zur Gesch\u00e4ftsordnung", 
    #                "Nachtrag: Beratung", "Nachtrag: M\u00fcndliche Frage/Schriftliche Antwort", 
    #                "Nachtrag: Regierungserkl\u00e4rung und Aussprache", "Nachtr\u00e4gliche \u00dcberweisung", 
    #                "Nachtr\u00e4gliche \u00dcberweisung gem\u00e4\u00df \u00a7 80 Gesch\u00e4ftsordnung BT", 
    #                "Neuabdruck der Antwort", "Nichtbeantwortung", "Ordnungsruf/Ordnungsgeld/Sitzungsausschluss", 
    #                "Plenarantrag", "Protokollerkl\u00e4rung/Begleiterkl\u00e4rung zum Vermittlungsverfahren", 
    #                "Regierungserkl\u00e4rung und Aussprache", "R\u00fccknahme", "R\u00fccknahme der Verordnung", 
    #                "R\u00fccknahme der Vorlage", "Schriftliche Frage zur m\u00fcndlichen Beantwortung wegen Frist\u00fcberschreitung", 
    #                "Schriftliche Frage/Schriftliche Antwort", "Unterrichtung", "Unterrichtung (Stellungnahme)", 
    #                "Unterrichtung durch den Vermittlungsausschuss", 
    #                "Unterrichtung \u00fcber Anrufung des Vermittlungsausschusses durch den BR", 
    #                "Unterrichtung \u00fcber Anrufung des Vermittlungsausschusses durch die BRg", 
    #                "Unterrichtung \u00fcber Beschluss des BT", "Unterrichtung \u00fcber Gesetzesbeschluss des BT", 
    #                "Unterrichtung \u00fcber Stellungnahme der BRg", "Unterrichtung \u00fcber Stellungnahme des BR", 
    #                "Unterrichtung \u00fcber Stellungnahme des BR und Gegen\u00e4u\u00dferung der BRg", 
    #                "Unterrichtung \u00fcber Zustimmungsversagung durch den BR", "Unterrichtung: Keine Ausschuss\u00fcberweisung", 
    #                "Vereidigung", "Verk\u00fcndung", "Vermittlungsvorschlag", "Verordnung", "Verordnungsantrag", 
    #                "Vertrauensantrag nach Art. 68 Grundgesetz", "Vorschlag", "Vorschlag / Vorlage / Mitteilung", "Wahl", 
    #                "Wahl des Bundespr\u00e4sidenten durch die Bundesversammlung", "Wahl des Pr\u00e4sidiums", 
    #                "Wahl des Vorsitzenden der Europakammer", "Wahlvorschlag", 
    #                "Zugeh\u00f6riger Vorgang: Antrag zur Mitwirkung des BT in EU-Angelegenheiten", 
    #                "Zur\u00fcckverweisung an die Aussch\u00fcsse", "Zur\u00fcckverweisung an die Aussch\u00fcsse in 2./3. Beratung", 
    #                "abstract", "aktivitaet_anzahl", "aktivitaet_anzeige", "aktualisiert", "ausfertigungsdatum", 
    #                "beschlussfassung", "datum", "dokumentart", "einleitungstext", "erlaeuterung", "fortsetzung", "fundstelle", 
    #                "fundstelle.anfangsquadrant", "fundstelle.anfangsseite", "fundstelle.anlagen", "fundstelle.datum", 
    #                "fundstelle.dokumentart", "fundstelle.dokumentnummer", "fundstelle.drucksachetyp", "fundstelle.endquadrant", 
    #                "fundstelle.endseite", "fundstelle.frage_nummer", "fundstelle.herausgeber", "fundstelle.id", "fundstelle.pdf_url", 
    #                "fundstelle.top", "fundstelle.top_zusatz", "fundstelle.urheber", "fundstelle.verteildatum", "fundstelle.xml_url", 
    #                "gang", "heftnummer", "id", "jahrgang", "kom", "mitberaten", "nachtrag", "pdf_url", "ratsdok", "ressort", 
    #                "rubrik_nr", "seite", "titel", "typ", "ueberweisung", "urheber", "verkuendungsblatt_bezeichnung", 
    #                "verkuendungsblatt_kuerzel", "verkuendungsdatum", "vorgang_id", "vorgangsposition", "vorgangstyp", 
    #                "zuordnung", "\u00c4nderung der Ausschuss\u00fcberweisung", "\u00c4nderungsantrag", 
    #                "\u00dcberweisung gem\u00e4\u00df \u00a7 80 Abs. 3 Gesch\u00e4ftsordnung BT", 
    #                "\u00dcberweisung gem\u00e4\u00df \u00a7 92 Gesch\u00e4ftsordnung BT", 
    #                "\u00dcberweisung gem\u00e4\u00df \u00a7 93 Gesch\u00e4ftsordnung BT", 
    #                "\u00dcberweisung gem\u00e4\u00df \u00a7 93 Gesch\u00e4ftsordnung BT: \u00c4nderung der Ausschuss\u00fcberweisung"],
    #     "objects": ["Aktuelle Stunde", "Allgemeine Verwaltungsvorschrift", "Anmerkung zur Plenarsitzung", 
    #                 "Ansprache/Erkl\u00e4rung/Mitteilung", "Antrag", "Befragung der Bundesregierung", "Bericht, Gutachten, Programm", 
    #                 "Besetzung externer Gremien durch BR", "Besetzung externer Gremien durch BT", "Besetzung interner Gremien des BT", 
    #                 "Bestellung und Abberufung von MdBR und stellv. MdBR", "EU-Vorlage", "Enquete-Kommission", 
    #                 "Entlastung der Bundesregierung", "Entlastung des Bundesrechnungshofes", "Entschlie\u00dfungsantrag BT", 
    #                 "Gesch\u00e4ftsordnung", "Gesetzgebung", "Gro\u00dfe Anfrage", "Immunit\u00e4tsangelegenheit", "Kleine Anfrage", 
    #                 "M\u00fcndliche Frage", "Parlamentarische Sonderkommission", "Petition", "Rechtsverordnung", 
    #                 "Rechtsverordnung (Au\u00dfenwirtschaftsrecht)", "Regierungserkl\u00e4rung", "Schriftliche Frage", 
    #                 "Selbst\u00e4ndiger Antrag von L\u00e4ndern auf Entschlie\u00dfung", 
    #                 "Unterrichtung durch das Europ\u00e4ische Parlament", "Untersuchungsausschuss", "Vereinbarte Debatte", 
    #                 "Verfahren vor dem Bundesverfassungsgericht", "Verordnungsantrag der L\u00e4nder", 
    #                 "Vertrauensantrag des Bundeskanzlers", "Wahl der Ausschussvorsitzenden BR", 
    #                 "Wahl der Richter des Bundesverfassungsgerichts ", "Wahl der Schriftf\u00fchrer BR", 
    #                 "Wahl des Bundespr\u00e4sidenten", "Wahl des Pr\u00e4sidiums BR", "Wahl im BT", "Wahlperiodenwechsel", 
    #                 "Wahlpr\u00fcfungsverfahren", "abstract", "aktualisiert", "archiv", "beratungsstand", "datum", "deskriptor", 
    #                 "gesta", "id", "initiative", "inkrafttreten", "kom", "mitteilung", "ratsdok", "sachgebiet", "titel", "typ", 
    #                 "verkuendung", "vorgang_verlinkung", "vorgangstyp", "wahlperiode", "zustimmungsbeduerftigkeit", 
    #                 "\u00dcber- und au\u00dferplanm\u00e4\u00dfige Haushaltsausgaben"]
    # },
    "03_hiring": {
        "events": ["Approve Job Requisition", "Attend Orientation Training", "Candidate Accept Offer", "Candidate Reject Offer", 
                   "Complete Assessment", "Complete Background Check", "Complete HR Paperwork", "Complete Onboarding", 
                   "Conduct Initial Interview", "Conduct Panel Interview", "Create Job Requisition", "Evaluate Interview", 
                   "Initiate Background Check", "Post Job Opening", "Prepare Offer Letter", "Review Assessment", 
                   "Schedule Initial Interview", "Schedule Panel Interview", "Screen Resume", "Send Assessment", "Send Offer Letter", 
                   "Send Rejection Letter", "Start Onboarding", "Submit Application"],
        "objects": ["Assessment", "Candidate", "HiringManager", "Interviewer", "JobRequisition", "Recruiter"]
    },
    "reasoning_benchmark": {
        "events": ["Abductive Reasoning - IND", "Abductive Reasoning - NE", "Abductive Reasoning - PE", "Backtracking - IND", 
                   "Backtracking - NE", "Backtracking - PE", "Conclusion - C", "Conclusion - PC", "Conclusion - W", 
                   "Counterfactual Reasoning - IND", "Counterfactual Reasoning - NE", "Counterfactual Reasoning - PE", 
                   "Deductive Reasoning - IND", "Deductive Reasoning - NE", "Deductive Reasoning - PE", 
                   "Ethical or Moral Reasoning - IND", "Ethical or Moral Reasoning - PE", "Heuristic Reasoning - IND", 
                   "Heuristic Reasoning - NE", "Heuristic Reasoning - PE", "Hypothesis Generation - IND", "Hypothesis Generation - NE", 
                   "Hypothesis Generation - PE", "Inductive Reasoning - IND", "Inductive Reasoning - NE", "Inductive Reasoning - PE", 
                   "Pattern Recognition - IND", "Pattern Recognition - NE", "Pattern Recognition - PE", "Validation - IND", 
                   "Validation - NE", "Validation - PE", "text"],
        "objects": ["MOD", "MODQUE"]
    },
    # too big for now, but can be added later
    # "age_of_empires2": {
    #     "events": ["Command Queue Arambai", "Command Queue Archer", "Command Queue Armored Elephant", "Command Queue Ballista Elephant", "Command Queue Battering Ram", "Command Queue Battle Elephant", "Command Queue Berserk", "Command Queue Bombard Cannon", "Command Queue Boyar", "Command Queue Camel Archer", "Command Queue Camel Scout", "Command Queue Cannon Galleon", "Command Queue Caravel", "Command Queue Cataphract", "Command Queue Cavalry Archer", "Command Queue Chakram Thrower", "Command Queue Chu Ko Nu", "Command Queue Conquistador", "Command Queue Coustillier", "Command Queue Demolition Raft", "Command Queue Eagle Scout", "Command Queue Elephant Archer", "Command Queue Fire Galley", "Command Queue Fishing Ship", "Command Queue Galley", "Command Queue Gbeto", "Command Queue Genitour", "Command Queue Genoese Crossbowman", "Command Queue Hand Cannoneer", "Command Queue Huskarl", "Command Queue Hussite Wagon", "Command Queue Jaguar Warrior", "Command Queue Janissary", "Command Queue Kamayuk", "Command Queue Karambit Warrior", "Command Queue Keshik", "Command Queue Knight", "Command Queue Konnik", "Command Queue Leitis", "Command Queue Longbowman", "Command Queue Magyar Huszar", "Command Queue Mameluke", "Command Queue Mangonel", "Command Queue Mangudai", "Command Queue Militia", "Command Queue Missionary", "Command Queue Monk", "Command Queue None", "Command Queue Obuch", "Command Queue Organ Gun", "Command Queue Petard", "Command Queue Plumed Archer", "Command Queue Ratha", "Command Queue Rattan Archer", "Command Queue Samurai", "Command Queue Scorpion", "Command Queue Scout Cavalry", "Command Queue Shotel Warrior", "Command Queue Shrivamsha Rider", "Command Queue Siege Tower", "Command Queue Skirmisher", "Command Queue Spearman", "Command Queue Steppe Lancer", "Command Queue Tarkan", "Command Queue Teutonic Knight", "Command Queue Throwing Axeman", "Command Queue Trade Cart", "Command Queue Transport Ship", "Command Queue Trebuchet", "Command Queue Villager", "Command Queue War Elephant", "Command Queue War Wagon", "Command Queue Warrior Priest", "Command Research ", "Command Research Anarchy", "Command Research Andean Sling", "Command Research Arbalester", "Command Research Architecture", "Command Research Arquebus", "Command Research Arrowslits", "Command Research Arson", "Command Research Artillery", "Command Research Atlatl", "Command Research Atonement", "Command Research Bagains", "Command Research Ballistics", "Command Research Bearded Axe", "Command Research Blast Furnace", "Command Research Block Printing", "Command Research Bloodlines", "Command Research Bodkin Arrow", "Command Research Bombard Tower", "Command Research Bow Saw", "Command Research Bracer", "Command Research Burgundian Vineyards", "Command Research Capped Ram", "Command Research Castle Age", "Command Research Cavalier", "Command Research Chain Barding Armor", "Command Research Chain Mail Armor", "Command Research Champion", "Command Research Chemistry", "Command Research Chieftains", "Command Research Chivalry", "Command Research Conscription", "Command Research Corvinian Army", "Command Research Crenellations", "Command Research Crossbowman", "Command Research DoubleBit Axe", "Command Research Drill", "Command Research Druzhina", "Command Research Eagle Warrior", "Command Research Elite Arambai", "Command Research Elite Battle Elephant", "Command Research Elite Berserk", "Command Research Elite Boyar", "Command Research Elite Camel Archer", "Command Research Elite Cataphract", "Command Research Elite Chu Ko Nu", "Command Research Elite Conquistador", "Command Research Elite Eagle Warrior", "Command Research Elite Elephant Archer", "Command Research Elite Gbeto", "Command Research Elite Genitour", "Command Research Elite Hussite Wagon", "Command Research Elite Janissary", "Command Research Elite Kamayuk", "Command Research Elite Karambit Warrior", "Command Research Elite Leitis", "Command Research Elite Longbowman", "Command Research Elite Magyar Huszar", "Command Research Elite Mangudai", "Command Research Elite Obuch", "Command Research Elite Organ Gun", "Command Research Elite Rattan Archer", "Command Research Elite Samurai", "Command Research Elite Shotel Warrior", "Command Research Elite Skirmisher", "Command Research Elite Steppe Lancer", "Command Research Elite Teutonic Knight", "Command Research Elite Throwing Axeman", "Command Research Elite War Elephant", "Command Research Eupseong", "Command Research Fabric Shields", "Command Research Farimba", "Command Research Fervor", "Command Research Feudal Age", "Command Research First Crusade", "Command Research Fletching", "Command Research Forced Levy", "Command Research Forging", "Command Research Fortified Wall", "Command Research Garland Wars", "Command Research Gillnets", "Command Research Gold Mining", "Command Research Gold Shaft Mining", "Command Research Guard Tower", "Command Research Guilds", "Command Research Halberdier", "Command Research Hand Cart", "Command Research Heated Shot", "Command Research Heavy Camel Rider", "Command Research Heavy Cav Archer", "Command Research Heavy Plow", "Command Research Heavy Scorpion", "Command Research Herbal Medicine", "Command Research Hill Forts", "Command Research Hoardings", "Command Research Horse Collar", "Command Research Houfnice", "Command Research Howdah", "Command Research Husbandry", "Command Research Hussar", "Command Research Illumination", "Command Research Imperial Age", "Command Research Inquisition", "Command Research Iron Casting", "Command Research Ironclad", "Command Research Kamandaran", "Command Research Kasbah", "Command Research Kataparuto", "Command Research Keep", "Command Research Leather Archer Armor", "Command Research Light Cavalry", "Command Research Logistica", "Command Research Long Swordsman", "Command Research Loom", "Command Research Maghrebi Camels", "Command Research ManatArms", "Command Research Manipur Cavalry", "Command Research Masonry", "Command Research Murder Holes", "Command Research None", "Command Research Onager", "Command Research Padded Archer Armor", "Command Research Paladin", "Command Research Paper Money", "Command Research Parthian Tactics", "Command Research Pavise", "Command Research Perfusion", "Command Research Pikeman", "Command Research Plate Barding Armor", "Command Research Plate Mail Armor", "Command Research Recurve Bow", "Command Research Redemption", "Command Research Ring Archer Armor", "Command Research Rocketry", "Command Research Royal Heirs", "Command Research Sanctity", "Command Research Sappers", "Command Research Scale Barding Armor", "Command Research Scale Mail Armor", "Command Research Scutage", "Command Research Shinkichon", "Command Research Siege Engineers", "Command Research Siege Onager", "Command Research Siege Ram", "Command Research Silk Armor", "Command Research Sipahi", "Command Research Spies/Treason", "Command Research Squires", "Command Research Steppe Husbandry", "Command Research Stone Mining", "Command Research Stone Shaft Mining", "Command Research Supplies", "Command Research Supremacy", "Command Research Theocracy", "Command Research Thumb Ring", "Command Research Tigui", "Command Research Timurid Siegecraft", "Command Research Torsion Engines", "Command Research Tower Shields", "Command Research Town Patrol", "Command Research Town Watch", "Command Research Treadmill Crane", "Command Research Tusk Swords", "Command Research TwoHanded Swordsman", "Command Research War Galley", "Command Research Warwolf", "Command Research Wheelbarrow", "Command Research Winged Hussar", "Command Research Yasama", "Command Research Yeomen", "Complete Build Archery Range", "Complete Build Barracks", "Complete Build Blacksmith", "Complete Build Bombard Tower", "Complete Build Castle", "Complete Build Dock", "Complete Build Donjon", "Complete Build Farm", "Complete Build Feitoria", "Complete Build Fish Trap", "Complete Build Gate", "Complete Build House", "Complete Build Krepost", "Complete Build Lumber Camp", "Complete Build Market", "Complete Build Mill", "Complete Build Mining Camp", "Complete Build Monastery", "Complete Build None", "Complete Build Outpost", "Complete Build Palisade Gate", "Complete Build Siege Workshop", "Complete Build Stable", "Complete Build Town Center", "Complete Build University", "Complete Build Watch Tower", "Complete Queue Arambai", "Complete Queue Archer", "Complete Queue Armored Elephant", "Complete Queue Ballista Elephant", "Complete Queue Battering Ram", "Complete Queue Battle Elephant", "Complete Queue Berserk", "Complete Queue Bombard Cannon", "Complete Queue Boyar", "Complete Queue Camel Archer", "Complete Queue Camel Scout", "Complete Queue Cannon Galleon", "Complete Queue Caravel", "Complete Queue Cataphract", "Complete Queue Cavalry Archer", "Complete Queue Chakram Thrower", "Complete Queue Chu Ko Nu", "Complete Queue Conquistador", "Complete Queue Coustillier", "Complete Queue Demolition Raft", "Complete Queue Eagle Scout", "Complete Queue Elephant Archer", "Complete Queue Fire Galley", "Complete Queue Fishing Ship", "Complete Queue Galley", "Complete Queue Gbeto", "Complete Queue Genitour", "Complete Queue Genoese Crossbowman", "Complete Queue Hand Cannoneer", "Complete Queue Huskarl", "Complete Queue Hussite Wagon", "Complete Queue Jaguar Warrior", "Complete Queue Janissary", "Complete Queue Kamayuk", "Complete Queue Karambit Warrior", "Complete Queue Keshik", "Complete Queue Knight", "Complete Queue Konnik", "Complete Queue Leitis", "Complete Queue Longbowman", "Complete Queue Magyar Huszar", "Complete Queue Mameluke", "Complete Queue Mangonel", "Complete Queue Mangudai", "Complete Queue Militia", "Complete Queue Missionary", "Complete Queue Monk", "Complete Queue None", "Complete Queue Obuch", "Complete Queue Organ Gun", "Complete Queue Petard", "Complete Queue Plumed Archer", "Complete Queue Ratha", "Complete Queue Rattan Archer", "Complete Queue Samurai", "Complete Queue Scorpion", "Complete Queue Scout Cavalry", "Complete Queue Shotel Warrior", "Complete Queue Shrivamsha Rider", "Complete Queue Siege Tower", "Complete Queue Skirmisher", "Complete Queue Spearman", "Complete Queue Steppe Lancer", "Complete Queue Tarkan", "Complete Queue Teutonic Knight", "Complete Queue Throwing Axeman", "Complete Queue Trade Cart", "Complete Queue Transport Ship", "Complete Queue Trebuchet", "Complete Queue Villager", "Complete Queue War Elephant", "Complete Queue War Wagon", "Complete Queue Warrior Priest", "Complete Research ", "Complete Research Anarchy", "Complete Research Andean Sling", "Complete Research Arbalester", "Complete Research Architecture", "Complete Research Arquebus", "Complete Research Arrowslits", "Complete Research Arson", "Complete Research Artillery", "Complete Research Atlatl", "Complete Research Atonement", "Complete Research Bagains", "Complete Research Ballistics", "Complete Research Bearded Axe", "Complete Research Blast Furnace", "Complete Research Block Printing", "Complete Research Bloodlines", "Complete Research Bodkin Arrow", "Complete Research Bombard Tower", "Complete Research Bow Saw", "Complete Research Bracer", "Complete Research Burgundian Vineyards", "Complete Research Capped Ram", "Complete Research Castle Age", "Complete Research Cavalier", "Complete Research Chain Barding Armor", "Complete Research Chain Mail Armor", "Complete Research Champion", "Complete Research Chemistry", "Complete Research Chieftains", "Complete Research Chivalry", "Complete Research Conscription", "Complete Research Corvinian Army", "Complete Research Crenellations", "Complete Research Crossbowman", "Complete Research DoubleBit Axe", "Complete Research Drill", "Complete Research Druzhina", "Complete Research Eagle Warrior", "Complete Research Elite Arambai", "Complete Research Elite Battle Elephant", "Complete Research Elite Berserk", "Complete Research Elite Boyar", "Complete Research Elite Camel Archer", "Complete Research Elite Cataphract", "Complete Research Elite Chu Ko Nu", "Complete Research Elite Conquistador", "Complete Research Elite Eagle Warrior", "Complete Research Elite Elephant Archer", "Complete Research Elite Gbeto", "Complete Research Elite Genitour", "Complete Research Elite Hussite Wagon", "Complete Research Elite Janissary", "Complete Research Elite Kamayuk", "Complete Research Elite Karambit Warrior", "Complete Research Elite Leitis", "Complete Research Elite Longbowman", "Complete Research Elite Magyar Huszar", "Complete Research Elite Mangudai", "Complete Research Elite Obuch", "Complete Research Elite Organ Gun", "Complete Research Elite Rattan Archer", "Complete Research Elite Samurai", "Complete Research Elite Shotel Warrior", "Complete Research Elite Skirmisher", "Complete Research Elite Steppe Lancer", "Complete Research Elite Teutonic Knight", "Complete Research Elite Throwing Axeman", "Complete Research Elite War Elephant", "Complete Research Eupseong", "Complete Research Fabric Shields", "Complete Research Farimba", "Complete Research Fervor", "Complete Research Feudal Age", "Complete Research First Crusade", "Complete Research Fletching", "Complete Research Forced Levy", "Complete Research Forging", "Complete Research Fortified Wall", "Complete Research Garland Wars", "Complete Research Gillnets", "Complete Research Gold Mining", "Complete Research Gold Shaft Mining", "Complete Research Guard Tower", "Complete Research Guilds", "Complete Research Halberdier", "Complete Research Hand Cart", "Complete Research Heated Shot", "Complete Research Heavy Camel Rider", "Complete Research Heavy Cav Archer", "Complete Research Heavy Plow", "Complete Research Heavy Scorpion", "Complete Research Herbal Medicine", "Complete Research Hill Forts", "Complete Research Hoardings", "Complete Research Horse Collar", "Complete Research Houfnice", "Complete Research Howdah", "Complete Research Husbandry", "Complete Research Hussar", "Complete Research Illumination", "Complete Research Imperial Age", "Complete Research Inquisition", "Complete Research Iron Casting", "Complete Research Ironclad", "Complete Research Kamandaran", "Complete Research Kasbah", "Complete Research Kataparuto", "Complete Research Keep", "Complete Research Leather Archer Armor", "Complete Research Light Cavalry", "Complete Research Logistica", "Complete Research Long Swordsman", "Complete Research Loom", "Complete Research Maghrebi Camels", "Complete Research ManatArms", "Complete Research Manipur Cavalry", "Complete Research Masonry", "Complete Research Murder Holes", "Complete Research None", "Complete Research Onager", "Complete Research Padded Archer Armor", "Complete Research Paladin", "Complete Research Paper Money", "Complete Research Parthian Tactics", "Complete Research Pavise", "Complete Research Perfusion", "Complete Research Pikeman", "Complete Research Plate Barding Armor", "Complete Research Plate Mail Armor", "Complete Research Recurve Bow", "Complete Research Redemption", "Complete Research Ring Archer Armor", "Complete Research Rocketry", "Complete Research Royal Heirs", "Complete Research Sanctity", "Complete Research Sappers", "Complete Research Scale Barding Armor", "Complete Research Scale Mail Armor", "Complete Research Scutage", "Complete Research Shinkichon", "Complete Research Siege Engineers", "Complete Research Siege Onager", "Complete Research Siege Ram", "Complete Research Silk Armor", "Complete Research Sipahi", "Complete Research Spies/Treason", "Complete Research Squires", "Complete Research Steppe Husbandry", "Complete Research Stone Mining", "Complete Research Stone Shaft Mining", "Complete Research Supplies", "Complete Research Supremacy", "Complete Research Theocracy", "Complete Research Thumb Ring", "Complete Research Tigui", "Complete Research Timurid Siegecraft", "Complete Research Torsion Engines", "Complete Research Tower Shields", "Complete Research Town Patrol", "Complete Research Town Watch", "Complete Research Treadmill Crane", "Complete Research Tusk Swords", "Complete Research TwoHanded Swordsman", "Complete Research War Galley", "Complete Research Warwolf", "Complete Research Wheelbarrow", "Complete Research Winged Hussar", "Complete Research Yasama", "Complete Research Yeomen", "Create Initial Archery Range", "Create Initial Bactrian Camel", "Create Initial Barracks", "Create Initial Camel Scout", "Create Initial Castle", "Create Initial Eagle Scout", "Create Initial Farm", "Create Initial Gate", "Create Initial Gate down", "Create Initial Gate up", "Create Initial Goat", "Create Initial Goose", "Create Initial Horse A", "Create Initial Horse B", "Create Initial Horse C", "Create Initial Horse E", "Create Initial House", "Create Initial Llama", "Create Initial Market", "Create Initial Outpost", "Create Initial Palisade Gate", "Create Initial Palisade Gate down", "Create Initial Palisade Gate up", "Create Initial Palisade Wall", "Create Initial Pig", "Create Initial Scout Cavalry", "Create Initial Sea Tower", "Create Initial Sheep", "Create Initial Stable", "Create Initial Stone Wall", "Create Initial Town Center", "Create Initial Transport Ship", "Create Initial Turkey", "Create Initial Villager", "Create Initial Watch Tower", "Create Initial Water Buffalo", "Gather Food", "Gather Gold", "Gather None", "Gather Relic", "Gather Stone", "Gather Wood", "Set Gather Point Food", "Set Gather Point Gold", "Set Gather Point None", "Set Gather Point Relic", "Set Gather Point Stone", "Set Gather Point Wood", "Start Build Archery Range", "Start Build Barracks", "Start Build Blacksmith", "Start Build Bombard Tower", "Start Build Castle", "Start Build Dock", "Start Build Donjon", "Start Build Farm", "Start Build Feitoria", "Start Build Fish Trap", "Start Build Gate", "Start Build House", "Start Build Krepost", "Start Build Lumber Camp", "Start Build Market", "Start Build Mill", "Start Build Mining Camp", "Start Build Monastery", "Start Build None", "Start Build Outpost", "Start Build Palisade Gate", "Start Build Siege Workshop", "Start Build Stable", "Start Build Town Center", "Start Build University", "Start Build Watch Tower", "Start Queue Arambai", "Start Queue Archer", "Start Queue Armored Elephant", "Start Queue Ballista Elephant", "Start Queue Battering Ram", "Start Queue Battle Elephant", "Start Queue Berserk", "Start Queue Bombard Cannon", "Start Queue Boyar", "Start Queue Camel Archer", "Start Queue Camel Scout", "Start Queue Cannon Galleon", "Start Queue Caravel", "Start Queue Cataphract", "Start Queue Cavalry Archer", "Start Queue Chakram Thrower", "Start Queue Chu Ko Nu", "Start Queue Conquistador", "Start Queue Coustillier", "Start Queue Demolition Raft", "Start Queue Eagle Scout", "Start Queue Elephant Archer", "Start Queue Fire Galley", "Start Queue Fishing Ship", "Start Queue Galley", "Start Queue Gbeto", "Start Queue Genitour", "Start Queue Genoese Crossbowman", "Start Queue Hand Cannoneer", "Start Queue Huskarl", "Start Queue Hussite Wagon", "Start Queue Jaguar Warrior", "Start Queue Janissary", "Start Queue Kamayuk", "Start Queue Karambit Warrior", "Start Queue Keshik", "Start Queue Knight", "Start Queue Konnik", "Start Queue Leitis", "Start Queue Longbowman", "Start Queue Magyar Huszar", "Start Queue Mameluke", "Start Queue Mangonel", "Start Queue Mangudai", "Start Queue Militia", "Start Queue Missionary", "Start Queue Monk", "Start Queue None", "Start Queue Obuch", "Start Queue Organ Gun", "Start Queue Petard", "Start Queue Plumed Archer", "Start Queue Ratha", "Start Queue Rattan Archer", "Start Queue Samurai", "Start Queue Scorpion", "Start Queue Scout Cavalry", "Start Queue Shotel Warrior", "Start Queue Shrivamsha Rider", "Start Queue Siege Tower", "Start Queue Skirmisher", "Start Queue Spearman", "Start Queue Steppe Lancer", "Start Queue Tarkan", "Start Queue Teutonic Knight", "Start Queue Throwing Axeman", "Start Queue Trade Cart", "Start Queue Transport Ship", "Start Queue Trebuchet", "Start Queue Villager", "Start Queue War Elephant", "Start Queue War Wagon", "Start Queue Warrior Priest", "Start Research ", "Start Research Anarchy", "Start Research Andean Sling", "Start Research Arbalester", "Start Research Architecture", "Start Research Arquebus", "Start Research Arrowslits", "Start Research Arson", "Start Research Artillery", "Start Research Atlatl", "Start Research Atonement", "Start Research Bagains", "Start Research Ballistics", "Start Research Bearded Axe", "Start Research Blast Furnace", "Start Research Block Printing", "Start Research Bloodlines", "Start Research Bodkin Arrow", "Start Research Bombard Tower", "Start Research Bow Saw", "Start Research Bracer", "Start Research Burgundian Vineyards", "Start Research Capped Ram", "Start Research Castle Age", "Start Research Cavalier", "Start Research Chain Barding Armor", "Start Research Chain Mail Armor", "Start Research Champion", "Start Research Chemistry", "Start Research Chieftains", "Start Research Chivalry", "Start Research Conscription", "Start Research Corvinian Army", "Start Research Crenellations", "Start Research Crossbowman", "Start Research DoubleBit Axe", "Start Research Drill", "Start Research Druzhina", "Start Research Eagle Warrior", "Start Research Elite Arambai", "Start Research Elite Battle Elephant", "Start Research Elite Berserk", "Start Research Elite Boyar", "Start Research Elite Camel Archer", "Start Research Elite Cataphract", "Start Research Elite Chu Ko Nu", "Start Research Elite Conquistador", "Start Research Elite Eagle Warrior", "Start Research Elite Elephant Archer", "Start Research Elite Gbeto", "Start Research Elite Genitour", "Start Research Elite Hussite Wagon", "Start Research Elite Janissary", "Start Research Elite Kamayuk", "Start Research Elite Karambit Warrior", "Start Research Elite Leitis", "Start Research Elite Longbowman", "Start Research Elite Magyar Huszar", "Start Research Elite Mangudai", "Start Research Elite Obuch", "Start Research Elite Organ Gun", "Start Research Elite Rattan Archer", "Start Research Elite Samurai", "Start Research Elite Shotel Warrior", "Start Research Elite Skirmisher", "Start Research Elite Steppe Lancer", "Start Research Elite Teutonic Knight", "Start Research Elite Throwing Axeman", "Start Research Elite War Elephant", "Start Research Eupseong", "Start Research Fabric Shields", "Start Research Farimba", "Start Research Fervor", "Start Research Feudal Age", "Start Research First Crusade", "Start Research Fletching", "Start Research Forced Levy", "Start Research Forging", "Start Research Fortified Wall", "Start Research Garland Wars", "Start Research Gillnets", "Start Research Gold Mining", "Start Research Gold Shaft Mining", "Start Research Guard Tower", "Start Research Guilds", "Start Research Halberdier", "Start Research Hand Cart", "Start Research Heated Shot", "Start Research Heavy Camel Rider", "Start Research Heavy Cav Archer", "Start Research Heavy Plow", "Start Research Heavy Scorpion", "Start Research Herbal Medicine", "Start Research Hill Forts", "Start Research Hoardings", "Start Research Horse Collar", "Start Research Houfnice", "Start Research Howdah", "Start Research Husbandry", "Start Research Hussar", "Start Research Illumination", "Start Research Imperial Age", "Start Research Inquisition", "Start Research Iron Casting", "Start Research Ironclad", "Start Research Kamandaran", "Start Research Kasbah", "Start Research Kataparuto", "Start Research Keep", "Start Research Leather Archer Armor", "Start Research Light Cavalry", "Start Research Logistica", "Start Research Long Swordsman", "Start Research Loom", "Start Research Maghrebi Camels", "Start Research ManatArms", "Start Research Manipur Cavalry", "Start Research Masonry", "Start Research Murder Holes", "Start Research None", "Start Research Onager", "Start Research Padded Archer Armor", "Start Research Paladin", "Start Research Paper Money", "Start Research Parthian Tactics", "Start Research Pavise", "Start Research Perfusion", "Start Research Pikeman", "Start Research Plate Barding Armor", "Start Research Plate Mail Armor", "Start Research Recurve Bow", "Start Research Redemption", "Start Research Ring Archer Armor", "Start Research Rocketry", "Start Research Royal Heirs", "Start Research Sanctity", "Start Research Sappers", "Start Research Scale Barding Armor", "Start Research Scale Mail Armor", "Start Research Scutage", "Start Research Shinkichon", "Start Research Siege Engineers", "Start Research Siege Onager", "Start Research Siege Ram", "Start Research Silk Armor", "Start Research Sipahi", "Start Research Spies/Treason", "Start Research Squires", "Start Research Steppe Husbandry", "Start Research Stone Mining", "Start Research Stone Shaft Mining", "Start Research Supplies", "Start Research Supremacy", "Start Research Theocracy", "Start Research Thumb Ring", "Start Research Tigui", "Start Research Timurid Siegecraft", "Start Research Torsion Engines", "Start Research Tower Shields", "Start Research Town Patrol", "Start Research Town Watch", "Start Research Treadmill Crane", "Start Research Tusk Swords", "Start Research TwoHanded Swordsman", "Start Research War Galley", "Start Research Warwolf", "Start Research Wheelbarrow", "Start Research Winged Hussar", "Start Research Yasama", "Start Research Yeomen"],
    #     "objects": ["Archery Range", "Barracks", "Blacksmith", "Bombard Tower", "Castle", "Dock", "Donjon", "Farm", "Feitoria", "Fish Trap", "Gate", "House", "Krepost", "Lumber Camp", "Market", "Match", "Mill", "Mining Camp", "Monastery", "None", "Outpost", "Palisade Gate", "Player", "Session", "Siege Workshop", "Stable", "Town Center", "University", "Villager", "Watch Tower"]
    # },
    "renting": {
        "events": ["Accept Late Payment", "Apply for Viewing Appointment", "Check Paperwork", "Evict Tenant", "Extensive Screening", 
                   "Hand In The Paperwork", "Issue Warning", "Miss Rent Payment", "Move In", "Pay Rent", "Reject Prospective Tenant", 
                   "Screen Prospective Tenant", "Set Appointment", "Sign Contract", "Tenant Cancels Appartment", "View The Property"],
        "objects": ["Age", "Case", "Person", "Yearsofeducation", "isCitizen", "isGender", "isGermanspeaking", "isMarried", 
                    "numOcc_AcceptLatePayment", "numOcc_ApplyforViewingAppointment", "numOcc_CheckPaperwork", "numOcc_EvictTenant", 
                    "numOcc_ExtensiveScreening", "numOcc_HandInThePaperwork", "numOcc_IssueWarning", "numOcc_MissRentPayment", 
                    "numOcc_MoveIn", "numOcc_PayRent", "numOcc_RejectProspectiveTenant", "numOcc_ScreenProspectiveTenant", 
                    "numOcc_SetAppointment", "numOcc_SignContract", "numOcc_TenantCancelsAppartment", "numOcc_ViewTheProperty", 
                    "overallProtected", "reworkedActivities", "sumBoolDiscrFactors", "throughputTime"]
    },
    "blockchain": {
        "events": ["accept contribution to crowdsourcer", "burn tokens", "burned tokens", "call and transfer Ether", 
                   "call for trusted transfer", "call to buy (public)", "call to buy complete sets", 
                   "call to buy complete sets with cash (public)", "call to cancel order", "call to check if container for fee token", 
                   "call to check if container for fee window", "call to check if container for reporting participant", 
                   "call to check if container for share token", "call to check if container market", "call to check if fork happened", 
                   "call to check if initialization happened", "call to check if known universe", "call to claim trading proceeds", 
                   "call to create", "call to create categorical market", "call to create dispute crowdsourcer", 
                   "call to create initial reporter", "call to create mailbox", "call to create map", "call to create market", 
                   "call to create order", "call to create scalar market", "call to create share token", "call to create yes-no-market", 
                   "call to decrement open interest", "call to fill best order", "call to fill best order with limit", 
                   "call to fill order", "call to find bounding orders", "call to get REP token", 
                   "call to get dispute round duration in seconds", "call to get or cache designated report no show bond", 
                   "call to get or cache designated reporter stake", "call to get or cache market creation cost", 
                   "call to get or cache reporting fee divisor", "call to get or cache validity bond", 
                   "call to get or create fee window before", "call to get or create next fee window", "call to get timestamp", 
                   "call to increment open interest", "call to initialize", "call to initialize crowdsourcer", 
                   "call to initiate a trade", "call to log REP token transferred", "call to log complete set purchase", 
                   "call to log complete sets sold", "call to log dispute crowdsourcer completion", 
                   "call to log dispute crowdsourcer contribution", "call to log dispute crowdsourcer redemption", 
                   "call to log dispute crowdsourcer tokens burned", "call to log dispute crowdsourcer tokens minted", 
                   "call to log fee token burning", "call to log fee token minted", "call to log fee window burned", 
                   "call to log fee window minted", "call to log fee window redeemed", "call to log initial report submission", 
                   "call to log market creation", "call to log order cancellation", "call to log order creation", 
                   "call to log order filling", "call to log reporter redemption", "call to log share token burned", 
                   "call to log share token minted", "call to log share tokens transfer", "call to log trading proceeds claim", 
                   "call to notify about dispute crowdsourcer creation", "call to purchase complete sets", "call to sell complete sets", 
                   "call to sell complete sets (public)", "call to trade with limit", "call without Ether transfer", "cancel order", 
                   "claim trading proceeds", "complete dispute", "contribute to dispute", "create crowdsourcer", "create dispute", 
                   "create disputer for crowdsourcer", "create fee window", "create market", "create new accounting instance", 
                   "create new contract", "create order", "create universe", "delegate call", 
                   "delegate call for trusted cancel order transfer", "delegate call for trusted fee window transfer", 
                   "delegate call for trusted fill order transfer", "delegate call for trusted market transfer",
                   "delegate call for trusted order transfer", "delegate call for trusted reporting participant transfer", 
                   "delegate call for trusted universe buy", "delegate call for trusted universe transfer", 
                   "delegate call to add an entry to a map", "delegate call to approve", "delegate call to assert balances", 
                   "delegate call to assert is not better price", "delegate call to assert is not worse price", 
                   "delegate call to burn fee window", "delegate call to buy Atto tokens", "delegate call to buy participant tokens", 
                   "delegate call to check for better price", "delegate call to check for container for share token", 
                   "delegate call to check if active", "delegate call to check if container for fee token", 
                   "delegate call to check if container for fee window", "delegate call to check if container for market", 
                   "delegate call to check if container for reporting participant", 
                   "delegate call to check if designated reporter showed", "delegate call to check if designated reporter was correct", 
                   "delegate call to check if fee window is over", "delegate call to check if market is finalized", 
                   "delegate call to check if universe is forking", "delegate call to check if worse price", 
                   "delegate call to check validity", "delegate call to contribute", "delegate call to create a categorical market", 
                   "delegate call to create scalar market", "delegate call to create shares", "delegate call to create yes-no-market", 
                   "delegate call to decrement open interest", "delegate call to decrement open interest from market", 
                   "delegate call to decrement total escrow", "delegate call to deposit Ether", 
                   "delegate call to deposit Ether for Cash token", "delegate call to derive market creator fee amount", 
                   "delegate call to derive payout distribution hash", "delegate call to destroy shares", 
                   "delegate call to do initial report", "delegate call to get REP token", "delegate call to get allowance", 
                   "delegate call to get amount of an order", "delegate call to get as address or zero", 
                   "delegate call to get balance", "delegate call to get best order ID", "delegate call to get better order ID", 
                   "delegate call to get crowdsourcer", "delegate call to get current fee window", "delegate call to get decimals", 
                   "delegate call to get denomination token", "delegate call to get designated reporter", 
                   "delegate call to get dispute round duration in seconds", "delegate call to get dispute threshold for fork", 
                   "delegate call to get end time", "delegate call to get fee token", "delegate call to get fee window", 
                   "delegate call to get finalization time", "delegate call to get forking market", "delegate call to get market", 
                   "delegate call to get market creator mailbox", "delegate call to get number of designated reporter no-shows", 
                   "delegate call to get number of incorrect designated report markets", "delegate call to get number of invalid markets", 
                   "delegate call to get number of markets", "delegate call to get number of participants", 
                   "delegate call to get number of ticks", "delegate call to get or cache designated reporter no-show-bond", 
                   "delegate call to get or cache designated reporter stake", "delegate call to get or cache market creation cost", 
                   "delegate call to get or cache next fee window", "delegate call to get or cache reporting fee divisor", 
                   "delegate call to get or cache validity bond", "delegate call to get or create fee window before", 
                   "delegate call to get order ID", "delegate call to get order creator", "delegate call to get order money escrowed", 
                   "delegate call to get order shares escrowed", "delegate call to get order type", "delegate call to get outcome", 
                   "delegate call to get payout distribution hash", "delegate call to get payout numerator", 
                   "delegate call to get price", "delegate call to get report time stamp", "delegate call to get reporting participant", 
                   "delegate call to get share token", "delegate call to get size", "delegate call to get stake", 
                   "delegate call to get start time", "delegate call to get the number of outcomes", 
                   "delegate call to get the total escrow", "delegate call to get total supply", "delegate call to get universe", 
                   "delegate call to get winning payout distribution hash", "delegate call to get winning payout numerator", 
                   "delegate call to get worse order ID", "delegate call to increment open interest", 
                   "delegate call to increment the total escrowed", "delegate call to initialize", "delegate call to liquidate losing", 
                   "delegate call to market", "delegate call to migrate", "delegate call to mint fee tokens", 
                   "delegate call to mint for reporting participant", "delegate call to record fill order", 
                   "delegate call to redeem", "delegate call to redeem for reporting participant", "delegate call to remove order", 
                   "delegate call to report", "delegate call to save order", "delegate call to set price", 
                   "delegate call to transfer token", "delegate call to transfer token on behalf of the owner", 
                   "delegate call to update statistics once market is finalized", "delegate call to withdraw Ether", 
                   "delegate call to withdraw Ether (2)", "delegate call to withdraw Ether if possible", "fill order", 
                   "finalize crowdsourcer", "finalize market", "give approval to transfer tokens", "initialize crowdsourcer", 
                   "mint tokens", "minted tokens", "purchase complete sets", "redeem as initial reporter", "redeem dispute crowdsourcer", 
                   "redeem fee window", "sell complete sets", "submit initial report", "transfer market", "transfer tokens", 
                   "transferred tokens", "undecoded event", "withdraw contribution to crowdsourcer", "withdraw fees from crowdsourcer", 
                   "withdraw proceeds from crowdsourcer"],
        "objects": ["CONTRACT", "EOA", "MARKET", "ORDER", "TOKEN"]
    },
    "lending": {
        "events": ["Application Rejected", "Appointment Denied", "Hand In Credit Appliaction", "Loan Denied", 
                   "Make Visit to Assess Colatteral", "Request Appointment", "Request Co-Signer On Loan", "Set Appointment", 
                   "Sign Loan Agreement", "Submit File to Underwriter", "Verify Borrowers Information", "skipped_examination"],
        "objects": ["Age", "Case", "Creditscore", "Person", "Yearsofeducation", "isCitizen", "isGender", 
                    "isGermanspeaking", "numOcc_ApplicationRejected", "numOcc_AppointmentDenied", "numOcc_HandInCreditAppliaction", 
                    "numOcc_LoanDenied", "numOcc_MakeVisittoAssessColatteral", "numOcc_RequestAppointment", 
                    "numOcc_RequestCo-SignerOnLoan", "numOcc_SetAppointment", "numOcc_SignLoanAgreement", "numOcc_SubmitFiletoUnderwriter", 
                    "numOcc_VerifyBorrowersInformation", "numOcc_skipped_examination", "overallProtected", "reworkedActivities", 
                    "sumBoolDiscrFactors", "throughputTime"]
    },
    # "01_o2c": {
    #     "events": ["Apply Discount", "Apply Payment", "Approve Order", "Approve Return", "Generate Invoice", "Generate Pick List", "Generate Shipping Label", "Handle Payment Discrepancy", "Issue Credit Note", "Issue Payment Reminder", "Pack Items", "Perform Credit Check", "Pick Items", "Process Return Request", "Receive Customer Order", "Receive Payment", "Reject Order", "Schedule Order Fulfillment", "Send Invoice to Customer", "Ship Order", "Update Inventory", "Validate Customer Order"],
    #     "objects": ["CreditNote", "Customer", "Dispute", "Invoice", "Order", "Payment", "Product", "Return", "Shipment"]
    # },
    # "04_hospital": {
    #     "events": ["Appointment Scheduling", "Assign Bed", "Diagnostic Test Order", "Initial Evaluation", "Insurance Verification", "Patient Arrival", "Perform Diagnostic Test", "Pre-Admission Consultation", "Registration"],
    #     "objects": ["Bed", "DiagnosticEquipment", "InsuranceCompany", "LabTechnician", "Nurse", "Patient", "Physician", "Room"]
    # },
    "container_logistics": {
        "events": ["Book Vehicles", "Bring to Loading Bay", "Collect Goods", "Create Transport Document", "Depart", "Drive to Terminal", 
                   "Load Truck", "Load to Vehicle", "Order Empty Containers", "Pick Up Empty Container", "Place in Stock", 
                   "Register Customer Order", "Reschedule Container", "Weigh"],
        "objects": ["AmountofContainers", "AmountofGoods", "AmountofHandlingUnits", "Container", "Customer Order", 
                    "DepartureDate", "Forklift", "Handling Unit", "Status", "Transport Document", "Truck", 
                    "Vehicle", "Weight"]
    },
}


In [3]:
def generate_example():
    pattern = random.choice(patterns)
    
    # Pick a domain first to ensure semantic consistency
    domain = random.choice(list(domains.keys()))
    domain_data = domains[domain]

    A = random.choice(domain_data["events"])
    # Ensure event A and B are different for better logical examples
    B = random.choice([e for e in domain_data["events"] if e != A])
    obj = random.choice(domain_data["objects"])
    t = random.choice([1, 6, 24, 72])

    text = pattern.replace("EVENT_A", A)\
                  .replace("EVENT_B", B)\
                  .replace("OBJECT", obj)\
                  .replace("TIME", str(t))

    # Dynamically inject constraints based on the pattern
    constraints = []
    filters = []
    if "before" in pattern or "eventually" in pattern:
        constraints = [{
            "type": "Filter",
            "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": None }
        }]
    elif "within TIME hours" in pattern:
        constraints = [{
            "type": "Filter",
            "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": t * 3600 }
        }]
    elif "must not occur" in pattern:
        # Requires a child edge with a NOT constraint to be completely valid OCPQ
        constraints = [{
            "type": "NOT",
            "child_names": ["edge_1"]
        }]
    elif "refer to the same OBJECT" in pattern:
        filters = [
            { "type": "O2E", "object": 0, "event": 0, "qualifier": None },
            { "type": "O2E", "object": 0, "event": 1, "qualifier": None }
        ]

    ocpq_tree = {
            "nodes": [
                {
                    "Box": [
                        {
                            "newEventVars": { "0": [A], "1": [B] },
                            "newObjectVars": { "0": [obj] },
                            "filters": filters,
                            "sizeFilters": [],
                            "constraints": constraints,
                            "evVarLabels": {},
                            "obVarLabels": {}
                        },
                        [] # IDs of any connected child nodes in the tree
                    ]
                }
            ],
            "edgeNames": []
    }

    return {"nl": text, "query": ocpq_tree}

dataset = []
seen = set()

while len(dataset) < 200:
    example = generate_example()
    # Use the natural language text to check for uniqueness
    if example["nl"] not in seen:
        seen.add(example["nl"])
        dataset.append(example)

with open("ocpq_dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

In [4]:
dataset[:5]

[{'nl': 'review application and submit application refer to the same candidates',
  'query': {'nodes': [{'Box': [{'newEventVars': {'0': ['review application'],
        '1': ['submit application']},
       'newObjectVars': {'0': ['candidates']},
       'filters': [{'type': 'O2E', 'object': 0, 'event': 0, 'qualifier': None},
        {'type': 'O2E', 'object': 0, 'event': 1, 'qualifier': None}],
       'sizeFilters': [],
       'constraints': [],
       'evVarLabels': {},
       'obVarLabels': {}},
      []]}],
   'edgeNames': []}},
 {'nl': 'After ship package, add item must not occur',
  'query': {'nodes': [{'Box': [{'newEventVars': {'0': ['ship package'],
        '1': ['add item']},
       'newObjectVars': {'0': ['items']},
       'filters': [],
       'sizeFilters': [],
       'constraints': [{'type': 'NOT', 'child_names': ['edge_1']}],
       'evVarLabels': {},
       'obVarLabels': {}},
      []]}],
   'edgeNames': []}},
 {'nl': 'create order and cancel order refer to the same items',